# Encrypted Quipu Test 50 — AES password

Inscribes a text quipu wrapped in AES-password encryption from **apocrypha** (single-key payer). The passphrase is entered live below.

## How this protocol works

A symmetric-AES wrapper around a plaintext quipu, with the AES key derived deterministically from a passphrase.

**1. Inner content.** Build a regular plaintext quipu (here, a 0x00 text quipu) — `(inner_header_bytes, inner_body_bytes)`. Nothing about it is encryption-aware; it's just bytes that happen to start with the `c1dd0001` magic.

**2. Key derivation.** The passphrase you type is hashed:
```
aes_key = SHA256(passphrase_utf8)
```
Same passphrase always produces the same key — that's how decryption works. No salt, no iteration count: this is a deliberate minimal-cryptography choice (see project memory).

**3. Length-prefixed framing.** Before encryption, the inner quipu's bytes are joined into one stream with a 2-byte big-endian header-length prefix:
```
framed = <header_len:2 BE> <inner_header> <inner_body>
```
Why this prefix? Outside encryption, the *strand boundary* in the diamond is the natural separator between header and body (cabeza strand = header, cuerpo strands = body). Inside encryption everything is one concatenated stream, so we need an explicit length field to recover the split after decryption.

**4. Encryption.** AES-CBC with the derived key over the framed bytes:
```
ciphertext = AES-CBC(aes_key, framed)
           = <16-byte IV> <PKCS7-padded encrypted body>
```

**5. Outer header.** The encrypted quipu's own header follows the v1 spec uniformly:
```
c1dd 0001  0e  <tone>  ae  <variant=0x01>  [|TITLE|]
            ^   ^      ^    ^              ^
            |   |      |    |              optional public outer label
            |   |      |    01 = password-derived key
            |   |      ae = AES sub-family
            |   tone defaults to 0x00 ordinary (no metadata leak)
            type = encrypted family
```

**6. What lands on chain.** A reader who walks the diamond sees:
- Outer header (~14–30 bytes): magic, type, tone, sub-family, variant, optional title.
- Outer body (the ciphertext): one opaque blob.

That's all the metadata a non-decrypter can see: "an AES-password encrypted quipu, optionally titled X."

**7. Decryption.** A reader with the passphrase:
1. Recomputes `aes_key = SHA256(passphrase)`.
2. Runs `AES-CBC-Decrypt(aes_key, ciphertext)` → recovers the framed bytes.
3. Reads the 2-byte length prefix → splits into `(inner_header, inner_body)`.
4. Checks `inner_header[:4] == c1dd 0001` — the magic acts as a free integrity check. AES-CBC produces gibberish under a wrong key, but gibberish never starts with this 4-byte sequence by coincidence (probability ~1 in 4 billion).
5. Dispatches to the inner type's normal parser (here, the text reader).

**Payer**: apocrypha (`mi_prv.enc`, address `D6zKNn…`). All four AES/keydrop notebooks (50/51/52/54) pay from here.

## Setup

In [2]:
import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import os, sys, json, time, secrets
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import cryptos
import colegio_tools as ct
from colegio_tools import _txid_of_serial
from text import build_text_quipu, read_text_quipu
from encrypted import (build_aes_quipu, build_ecies_quipu, build_keydrop_quipu,
                       read_encrypted_quipu, aggregate_privkey, aggregate_pubkey,
                       TONE_ORDINARY, TONE_AFFECTION, TONE_REVERENCE)
from coincurve import PrivateKey as CCPriv, PublicKey as CCPub

doge = cryptos.Doge()
TIP  = 5_000_000   # 0.05 DOGE per knot

In [3]:
LLAVES = '../../cinv/llaves'
INSCRIPTIONS_READY = os.path.join(REPO, 'inscriptions_ready')
os.makedirs(INSCRIPTIONS_READY, exist_ok=True)

def load_priv(name, password=''):
    enc = open(os.path.join(LLAVES, f'{name}_prv.enc'), 'rb').read()
    return ct.import_privKey_from_bytes(enc, password)

# Payer keys
priv_apo  = load_priv('mi')      # apocrypha (single-key payer)
priv_key1 = load_priv('key1')    # the other half of multiman 2-of-2

# Recipient-pubkey-only keys (loaded so we can derive their pubkeys for ECIES)
priv_test1 = load_priv('test1')
priv_test2 = load_priv('test2')
priv_test3 = load_priv('test3')

addr_apo = doge.privtoaddr(priv_apo.to_hex()[2:])

# multiman 2-of-2 multisig (mi + key1)
multiman = json.load(open(os.path.join(LLAVES, 'multiman_multisig.json')))
addr_multiman = multiman['address']
redeem_multiman_hex = multiman['redeem_script_hex']

def cc_priv(p): return CCPriv(bytes.fromhex(p.to_hex()[2:]))
def cc_pub_from_priv(p): return cc_priv(p).public_key

pub_apo   = cc_pub_from_priv(priv_apo)
pub_key1  = cc_pub_from_priv(priv_key1)
pub_test1 = cc_pub_from_priv(priv_test1)
pub_test2 = cc_pub_from_priv(priv_test2)
pub_test3 = cc_pub_from_priv(priv_test3)

print(f'apocrypha payer:        {addr_apo}')
print(f'multiman 2-of-2 payer:  {addr_multiman}  (mi + key1)')

apocrypha payer:        D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX
multiman 2-of-2 payer:  A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC  (mi + key1)


## Build the inner text quipu

In [4]:
inner_h, inner_b = build_text_quipu('Un secreto',
                                    'Te encuentro en las estrellas; te pierdo en el amanecer.',
                                    tone=TONE_AFFECTION)
print(f'inner: header {len(inner_h)} B, body {len(inner_b)} B')

inner: header 18 B, body 56 B


## Enter the passphrase

In [5]:
import getpass
PASSPHRASE = getpass.getpass('AES passphrase: ') #mayo25
print(f'passphrase entered ({len(PASSPHRASE)} chars)')

AES passphrase: ········
passphrase entered (6 chars)


## Wrap with AES password

In [6]:
outer_h, outer_b = build_aes_quipu(inner_h, inner_b, PASSPHRASE,
                                    title='Para una conocida', tone=TONE_ORDINARY)
print(f'outer header ({len(outer_h)} B): {outer_h.hex()}')
print(f'outer body   ({len(outer_b)} B)')

outer header (27 B): c1dd00010e00ae017c5061726120756e6120636f6e6f636964617c
outer body   (108 B)


## Header byte-by-byte

In [7]:
h = outer_h
print(f'  0..3  {h[:4].hex():<10s}  magic + version')
print(f'  4     {h[4]:02x}          type = 0x0e (encrypted)')
print(f'  5     {h[5]:02x}          tone = 0x{h[5]:02x} (ordinary)')
print(f'  6     {h[6]:02x}          sub_family = 0xae (AES)')
print(f'  7     {h[7]:02x}          variant = 0x{h[7]:02x} (password)')
print(f'  8+    {h[8:].hex():<10s}  outer title bytes')

  0..3  c1dd0001    magic + version
  4     0e          type = 0x0e (encrypted)
  5     00          tone = 0x00 (ordinary)
  6     ae          sub_family = 0xae (AES)
  7     01          variant = 0x01 (password)
  8+    7c5061726120756e6120636f6e6f636964617c  outer title bytes


## Inscribe — diamond pattern from apocrypha

In [8]:
N_BODY_STRANDS = 4
chunk = len(outer_b) // N_BODY_STRANDS
extra = len(outer_b) %  N_BODY_STRANDS
body_parts, i = [], 0
for k in range(N_BODY_STRANDS):
    sz = chunk + (1 if k < extra else 0)
    body_parts.append(outer_b[i:i+sz]); i += sz
strand_payloads = [outer_h] + body_parts
print(f'{len(strand_payloads)} strands; sizes: {[len(p) for p in strand_payloads]}')

5 strands; sizes: [27, 27, 27, 27, 27]


## UTXOs at apocrypha

In [9]:
utxos = ct.rpc_request('listunspent', [0, 9999999, [addr_apo]])
seed_inputs = [{'output': f"{u['txid']}:{u['vout']}", 'value': int(round(u['amount']*1e8))} for u in utxos]
total = sum(s['value'] for s in seed_inputs)
print(f'  {len(seed_inputs)} UTXO(s), total {total/1e8:.4f} DOGE')

  1 UTXO(s), total 13.6500 DOGE


## Phase I — root tx

In [10]:
priv_hex = priv_apo.to_hex()[2:]
n = len(strand_payloads)
per = (total - TIP) // n
remainder = (total - TIP) - per * n
strand_seeds = [per] * n
strand_seeds[0] += remainder
root_outputs = [{'value': s, 'address': addr_apo} for s in strand_seeds]
root_tx = doge.mktx(seed_inputs, root_outputs)
doge.signall(root_tx, priv_hex)
root_hex = cryptos.serialize(root_tx)
root_txid = _txid_of_serial(root_hex)
assert ct.rpc_request('sendrawtransaction', [root_hex]) == root_txid
print(f'root txid: {root_txid}')

root txid: 00109923db8b1004ac48470516af86d41e6cf26d8667c32be6c0b3edc0cdb664


In [15]:
import time
while True:
    info = ct.rpc_request('getrawtransaction', [root_txid, 1])
    confs = info.get('confirmations', 0)
    print(f'{time.strftime("%H:%M:%S")}  confirmations: {confs}')
    if confs >= 1:
        break
    time.sleep(15)
print('✓ confirmed')

18:59:40  confirmations: 0
18:59:55  confirmations: 1
✓ confirmed


## Phase II — precompute + broadcast strands

In [16]:
strands = []
for si, payload in enumerate(strand_payloads):
    cad = ct.CadenaAtom(prvkey=priv_hex, data=payload,
                         utxo_dct={'output': f'{root_txid}:{si}', 'value': strand_seeds[si]},
                         tip=TIP)
    cad.precompute()
    strands.append(cad)
    print(f'  strand {si}: {len(cad.txns)} knots')
for si, cad in enumerate(strands):
    for hex_tx, txid in zip(cad.txns, cad.txn_ids):
        assert ct.rpc_request('sendrawtransaction', [hex_tx]) == txid
    print(f'  strand {si} broadcast')

  strand 0: 1 knots
  strand 1: 1 knots
  strand 2: 1 knots
  strand 3: 1 knots
  strand 4: 1 knots
  strand 0 broadcast
  strand 1 broadcast
  strand 2 broadcast
  strand 3 broadcast
  strand 4 broadcast


## Wait for strand termini to confirm

In [17]:
start_h = ct.rpc_request('getblockcount')
termini = [c.txn_ids[-1] for c in strands]
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        confs = [ct.rpc_request('getrawtransaction', [t, 1]).get('confirmations', 0) for t in termini]
        print(f'  block {h}  ' + '  '.join(f's{i}:{c}' for i,c in enumerate(confs)))
        if all(c >= 1 for c in confs):
            print('✓ confirmed'); break
        start_h = h
    time.sleep(15)

  block 6213571  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6213572  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6213573  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6213574  s0:1  s1:1  s2:1  s3:1  s4:1
✓ confirmed


## Phase III — join tx

In [18]:
join_inputs = []
for si, cad in enumerate(strands):
    terminus_value = strand_seeds[si] - TIP * len(cad.txns)
    join_inputs.append({'output': f'{cad.txn_ids[-1]}:0', 'value': terminus_value})
join_total = sum(i['value'] for i in join_inputs)
join_tx = doge.mktx(join_inputs, [{'value': join_total - TIP, 'address': addr_apo}])
doge.signall(join_tx, priv_hex)
join_hex = cryptos.serialize(join_tx)
join_txid = _txid_of_serial(join_hex)
assert ct.rpc_request('sendrawtransaction', [join_hex]) == join_txid
print(f'join txid: {join_txid}')

join txid: 234f84e2c807fe48952652b3ab3edac58bfcb6b98a3d203bb13f8cdbec1e94d4


In [19]:
while True:
    info = ct.rpc_request('getrawtransaction', [join_txid, 1])
    confs = info.get('confirmations', 0)
    print(f'{time.strftime("%H:%M:%S")}  confirmations: {confs}')
    if confs >= 1:
        break
    time.sleep(15)
print('✓ confirmed')

19:04:32  confirmations: 0
19:04:47  confirmations: 0
19:05:02  confirmations: 0
19:05:17  confirmations: 0
19:05:32  confirmations: 0
19:05:47  confirmations: 0
19:06:02  confirmations: 0
19:06:17  confirmations: 0
19:06:32  confirmations: 0
19:06:47  confirmations: 0
19:07:02  confirmations: 0
19:07:17  confirmations: 0
19:07:32  confirmations: 0
19:07:47  confirmations: 0
19:08:02  confirmations: 0
19:08:17  confirmations: 0
19:08:32  confirmations: 0
19:08:47  confirmations: 0
19:09:02  confirmations: 1
✓ confirmed


## Read back from chain

In [20]:
spender_map = {}
for si, cad in enumerate(strands):
    spender_map[f'{root_txid}:{si}'] = cad.txn_ids[0]
    for ki in range(len(cad.txn_ids) - 1):
        spender_map[f'{cad.txn_ids[ki]}:0'] = cad.txn_ids[ki+1]
def walk(start):
    out, cur = '', start
    while True:
        n = spender_map.get(cur)
        if not n: return out
        raw = ct.rpc_request('getrawtransaction', [n, 1])
        op = next((ct.extract_op_return(v) for v in raw['vout'] if ct.extract_op_return(v)), None)
        if not op: return out
        out += op; cur = f'{n}:0'
rec_h = bytes.fromhex(walk(f'{root_txid}:0'))
rec_b = b''.join(bytes.fromhex(walk(f'{root_txid}:{si}')) for si in range(1, len(strands)))
assert rec_h == outer_h and rec_b == outer_b
print('✓ recovered byte-identical')

✓ recovered byte-identical


## Decrypt with passphrase

In [21]:
parsed = read_encrypted_quipu(rec_h, rec_b, key=PASSPHRASE)
assert parsed['inner_header'] == inner_h and parsed['inner_body'] == inner_b
print('✓ decrypted successfully')
inner = read_text_quipu(parsed['inner_header'], parsed['inner_body'])
print(f"  title: {inner['title']!r}")
print(f"  body:  {inner['body']!r}")

✓ decrypted successfully
  title: 'Un secreto'
  body:  'Te encuentro en las estrellas; te pierdo en el amanecer.'


In [23]:
print(parsed['inner_header'])

b'\xc1\xdd\x00\x01\x00\x01|Un secreto|'
